# 🏆 Gammafest Football Match Prediction — Model Pipeline V24

**Strategy:** V23 + Isotonic Calibration Stage 1 + LGB+CatBoost Ensemble Stage 2 + ZINB Full Integration

### 📦 File yang harus diupload ke Google Colab:
1. `dataset/train_final.csv`
2. `dataset/test_final.csv`
3. `dataset/train_meta.csv`
4. `dataset/test_meta.csv`
5. `dataset/test_ground_truth.csv`

> Path default: `./dataset/`

⏱ **Estimasi runtime:** ~6-9 menit di Google Colab
📊 **Target AW-MAE:** ~2.30–2.40

In [ ]:
# ============================================================
# CELL 1: Setup & Instalasi Dependensi
# ============================================================
!pip install -q catboost lightgbm scikit-learn pandas numpy scipy

import numpy as np
import pandas as pd
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.model_selection import cross_val_predict
from sklearn.calibration import CalibratedClassifierCV
from sklearn.isotonic import IsotonicRegression
from scipy.special import softmax, expit
from scipy.stats import nbinom
import warnings, time, os, json
warnings.filterwarnings('ignore')

print('✅ Setup complete!')
print(f'LightGBM: {lgb.__version__}')
print(f'NumPy: {np.__version__}')

In [ ]:
# ============================================================
# CELL 2: Path & Load Data
# ============================================================
# from google.colab import drive
# drive.mount('/content/drive')

DATASET_DIR = './dataset/'  # <<< GANTI JIKA PERLU

# Cek file
required_files = ['train_final.csv', 'test_final.csv', 'train_meta.csv', 'test_meta.csv', 'test_ground_truth.csv']
for f in required_files:
    filepath = os.path.join(DATASET_DIR, f)
    if os.path.exists(filepath):
        print(f'✅ {f} FOUND')
    else:
        print(f'❌ {f} NOT FOUND — Upload ke {DATASET_DIR}')

train_raw = pd.read_csv(os.path.join(DATASET_DIR, 'train_final.csv'))
test_raw = pd.read_csv(os.path.join(DATASET_DIR, 'test_final.csv'))
train_meta = pd.read_csv(os.path.join(DATASET_DIR, 'train_meta.csv'))
test_meta = pd.read_csv(os.path.join(DATASET_DIR, 'test_meta.csv'))
ground_truth = pd.read_csv(os.path.join(DATASET_DIR, 'test_ground_truth.csv'))

print(f'\nTrain: {train_raw.shape}, Test: {test_raw.shape}')

In [ ]:
# ============================================================
# CELL 3: Feature Engineering
# ============================================================
def feature_engineering_v24(train, test, train_meta, test_meta):
    train = train.merge(train_meta[['id', 'gender']], on='id', how='left')
    test = test.merge(test_meta[['id', 'gender']], on='id', how='left')
    
    train_men = train[train['gender'] == 'men'].copy()
    train_women = train[train['gender'] == 'women'].copy()
    test_men = test[test['gender'] == 'men'].copy()
    test_women = test[test['gender'] == 'women'].copy()
    
    print(f'Men train: {train_men.shape}, Women train: {train_women.shape}')
    print(f'Men test: {test_men.shape}, Women test: {test_women.shape}')
    
    exclude_cols = ['id', 'gender', 'team_goals', 'opp_goals', 'outcome']
    feature_cols = [c for c in train.columns if c not in exclude_cols and train[c].dtype in ['int64', 'float64', 'bool']]
    
    # Handle NaN/Inf
    for df in [train_men, train_women, test_men, test_women]:
        df.replace([np.inf, -np.inf], np.nan, inplace=True)
        for col in feature_cols:
            if col in df.columns and df[col].dtype in ['float64', 'int64']:
                df[col] = df[col].fillna(df[col].median())
    
    return train_men, train_women, test_men, test_women, feature_cols

train_men, train_women, test_men, test_women, FEATURE_COLS = feature_engineering_v24(train_raw, test_raw, train_meta, test_meta)
print(f'\nTotal features: {len(FEATURE_COLS)}')

## Model V24 — Isotonic Calibration + LGB+CatBoost Ensemble

**Improvements over V23:**
- Stage 1: Isotonic Regression calibration untuk outcome probabilitas
- Stage 2: Ensemble LightGBM + CatBoost (rata-rata probabilitas)
- ZINB full integration (scoring + calibration)

In [ ]:
# ============================================================
# CELL 4: Model Functions — V24
# ============================================================

def build_aw_mae_matrix(max_goals=5):
    n = max_goals + 1
    M = np.zeros((n * n, n * n))
    for i in range(n * n):
        true_team = i // n
        true_opp = i % n
        for j in range(n * n):
            pred_team = j // n
            pred_opp = j % n
            M[i, j] = abs(true_team - pred_team) + abs(true_opp - pred_opp)
    return M

AW_MAE_MATRIX = build_aw_mae_matrix(5)

def erm_predict(pmf):
    scores = pmf @ AW_MAE_MATRIX
    best_idx = np.argmin(scores, axis=1)
    return best_idx // 6, best_idx % 6

def predict_outcome_from_pmf(pmf):
    N = len(pmf)
    p_win, p_draw, p_lose = np.zeros(N), np.zeros(N), np.zeros(N)
    for i in range(6):
        for j in range(6):
            k = i * 6 + j
            if i > j:    p_win += pmf[:, k]
            elif i == j: p_draw += pmf[:, k]
            else:        p_lose += pmf[:, k]
    return np.column_stack([p_win, p_draw, p_lose])

# ------------------------------------------------------------------
# ZINB Functions
# ------------------------------------------------------------------
def fit_zinb_params(lambdas):
    mean_lambda = np.mean(lambdas)
    var_lambda = np.var(lambdas)
    if var_lambda > mean_lambda and mean_lambda > 0:
        r = mean_lambda**2 / (var_lambda - mean_lambda)
        p = mean_lambda / var_lambda
        r = max(r, 0.5)
        p = np.clip(p, 0.01, 0.99)
    else:
        r = 1.0
        p = 0.5
    p_zero_obs = np.mean(lambdas == 0)
    p_zero_nb = nbinom.pmf(0, r, p)
    if p_zero_obs > p_zero_nb:
        pi = (p_zero_obs - p_zero_nb) / (1 - p_zero_nb)
    else:
        pi = 0.0
    pi = np.clip(pi, 0.0, 0.5)
    return r, p, pi

def zinb_pmf(r, p_val, pi, max_goals=5):
    probs = np.zeros(max_goals + 1)
    for k in range(max_goals + 1):
        nb_prob = nbinom.pmf(k, r, p_val)
        if k == 0:
            probs[k] = pi + (1 - pi) * nb_prob
        else:
            probs[k] = (1 - pi) * nb_prob
    return probs / probs.sum()

# ------------------------------------------------------------------
# Isotonic Calibration
# ------------------------------------------------------------------
def calibrate_outcome_probs(outcome_proba, X_train, y_outcome_train, X_test):
    """Kalibrasi outcome probabilites dengan Isotonic Regression per class."""
    # outcome_proba adalah (N,3): [p_win, p_draw, p_lose]
    calibrated = np.zeros_like(outcome_proba)
    
    for cls in range(3):
        y_binary = (y_outcome_train == cls).astype(int)
        # Fit isotonic regression pada prediktif probabilitas vs actual
        iso = IsotonicRegression(out_of_bounds='clip', y_min=0.001, y_max=0.999)
        iso.fit(outcome_proba[:, cls], y_binary)
        calibrated[:, cls] = iso.transform(outcome_proba[:, cls])
    
    # Renormalize
    calibrated = calibrated / calibrated.sum(axis=1, keepdims=True)
    return calibrated

# ------------------------------------------------------------------
# Evaluasi
# ------------------------------------------------------------------
def compute_aw_mae(pred_team, pred_opp, true_team, true_opp):
    return np.mean(np.abs(pred_team - true_team) + np.abs(pred_opp - true_opp))

def compute_outcome_acc(pred_team, pred_opp, true_team, true_opp):
    pred_outcome = np.where(pred_team > pred_opp, 1, np.where(pred_team == pred_opp, 0, -1))
    true_outcome = np.where(true_team > true_opp, 1, np.where(true_team == true_opp, 0, -1))
    return np.mean(pred_outcome == true_outcome)

def compute_exact_acc(pred_team, pred_opp, true_team, true_opp):
    return np.mean((pred_team == true_team) & (pred_opp == true_opp))

# ------------------------------------------------------------------
# Main Training V24
# ------------------------------------------------------------------
def train_model_v24(X_train, y_outcome_train, y_team_train, y_opp_train,
                     X_test, gender='men'):
    """Train V24: V23 + Isotonic Calibration + LGB+CatBoost Ensemble."""
    
    T = 1.1 if gender == 'men' else 1.2
    
    # Remap outcome: -1,0,1 → 0,1,2
    y_outcome_cls = y_outcome_train.copy()
    if y_outcome_cls.min() < 0:
        y_outcome_cls = pd.Series(y_outcome_cls).map({1: 0, 0: 1, -1: 2}).values
    
    # Stage 1: Outcome Classifier (CatBoost)
    print(f'  Training Stage 1 (Outcome) for {gender}...')
    outcome_model = CatBoostClassifier(
        iterations=500, learning_rate=0.05, depth=7,
        loss_function='MultiClass',
        class_weights=[1.0, 1.5, 1.0],
        random_seed=42, verbose=100,
        task_type='GPU' if os.environ.get('COLAB_GPU', '') else 'CPU'
    )
    outcome_model.fit(X_train, y_outcome_cls)
    
    # Cross-val predictions untuk calibration (pada training set)
    print(f'  Generating CV predictions for calibration...')
    cv_proba_train = np.zeros((len(X_train), 3))
    # Sederhanakan: gunakan OOB-like approach dengan train/predict sendiri
    # Untuk full calibration, gunakan cross_val_predict
    from sklearn.model_selection import KFold
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    for train_idx, val_idx in kf.split(X_train):
        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr = y_outcome_cls[train_idx]
        model_cv = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=7,
                                       loss_function='MultiClass', random_seed=42,
                                       verbose=0, task_type='CPU')
        model_cv.fit(X_tr, y_tr)
        cv_proba_train[val_idx] = model_cv.predict_proba(X_val)
    
    # Kalibrasi isotonic pada training set
    print(f'  Applying Isotonic Calibration...')
    iso_models = []
    cv_proba_cal = np.zeros_like(cv_proba_train)
    for cls in range(3):
        y_binary = (y_outcome_cls == cls).astype(int)
        iso = IsotonicRegression(out_of_bounds='clip', y_min=0.001, y_max=0.999)
        iso.fit(cv_proba_train[:, cls], y_binary)
        iso_models.append(iso)
        cv_proba_cal[:, cls] = iso.transform(cv_proba_train[:, cls])
    cv_proba_cal = cv_proba_cal / cv_proba_cal.sum(axis=1, keepdims=True)
    
    # Predict pada test set + calibrate
    proba_outcome_raw = outcome_model.predict_proba(X_test)
    proba_outcome_cal = np.zeros_like(proba_outcome_raw)
    for cls in range(3):
        proba_outcome_cal[:, cls] = iso_models[cls].transform(proba_outcome_raw[:, cls])
    proba_outcome_cal = proba_outcome_cal / proba_outcome_cal.sum(axis=1, keepdims=True)
    
    # Temperature scaling
    proba_outcome = proba_outcome_cal / T
    proba_outcome = softmax(proba_outcome, axis=1)
    
    # Stage 2: Ensemble LGB + CatBoost untuk Team & Opp Goals
    print(f'  Training Stage 2 (Goals — LGB+CatBoost Ensemble) for {gender}...')
    
    # Team Goals — LightGBM
    lgb_team = lgb.LGBMClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=7,
        num_leaves=63, random_state=43, verbose=-1, n_jobs=-1
    )
    lgb_team.fit(X_train, y_team_train)
    proba_team_lgb = lgb_team.predict_proba(X_test)
    
    # Team Goals — CatBoost
    cb_team = CatBoostClassifier(
        iterations=500, learning_rate=0.05, depth=7,
        loss_function='MultiClass', random_seed=44,
        verbose=100, task_type='GPU' if os.environ.get('COLAB_GPU', '') else 'CPU'
    )
    cb_team.fit(X_train, y_team_train)
    proba_team_cb = cb_team.predict_proba(X_test)
    
    # Ensemble average
    proba_team = 0.5 * proba_team_lgb + 0.5 * proba_team_cb
    
    # Opp Goals — LightGBM
    lgb_opp = lgb.LGBMClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=7,
        num_leaves=63, random_state=45, verbose=-1, n_jobs=-1
    )
    lgb_opp.fit(X_train, y_opp_train)
    proba_opp_lgb = lgb_opp.predict_proba(X_test)
    
    # Opp Goals — CatBoost
    cb_opp = CatBoostClassifier(
        iterations=500, learning_rate=0.05, depth=7,
        loss_function='MultiClass', random_seed=46,
        verbose=100, task_type='GPU' if os.environ.get('COLAB_GPU', '') else 'CPU'
    )
    cb_opp.fit(X_train, y_opp_train)
    proba_opp_cb = cb_opp.predict_proba(X_test)
    
    # Ensemble average
    proba_opp = 0.5 * proba_opp_lgb + 0.5 * proba_opp_cb
    
    # Joint PMF via outer product
    N = len(X_test)
    pmf_joint = np.zeros((N, 36))
    for i in range(6):
        for j in range(6):
            k = i * 6 + j
            pmf_joint[:, k] = proba_team[:, i] * proba_opp[:, j]
    
    # Soft cascade renormalization
    pmf_reweighted = np.zeros((N, 36))
    for n in range(N):
        win_idx = [i*6+j for i in range(6) for j in range(6) if i > j]
        draw_idx = [i*6+j for i in range(6) for j in range(6) if i == j]
        lose_idx = [i*6+j for i in range(6) for j in range(6) if i < j]
        
        masks = [win_idx, draw_idx, lose_idx]
        for o, mask in enumerate(masks):
            p_joint = pmf_joint[n, mask].sum()
            if p_joint > 0:
                pmf_reweighted[n, mask] = pmf_joint[n, mask] * (proba_outcome[n, o] / p_joint)
    
    pmf_reweighted = pmf_reweighted / pmf_reweighted.sum(axis=1, keepdims=True)
    team_goals_pred, opp_goals_pred = erm_predict(pmf_reweighted)
    
    return {
        'pmf': pmf_reweighted,
        'proba_outcome': proba_outcome,
        'team_goals': team_goals_pred,
        'opp_goals': opp_goals_pred
    }

print('✅ Model V24 functions defined!')

In [ ]:
# ============================================================
# CELL 5: Train & Predict — MEN (V24)
# ============================================================
print('='*60)
print('TRAINING MEN MODEL (V24)')
print('='*60)

X_men_train = train_men[FEATURE_COLS].values.astype(np.float32)
y_outcome_men = train_men['outcome'].values
y_team_men = np.clip(train_men['team_goals'].values.astype(int), 0, 5)
y_opp_men = np.clip(train_men['opp_goals'].values.astype(int), 0, 5)

X_men_test = test_men[FEATURE_COLS].values.astype(np.float32)

t0 = time.time()
men_results = train_model_v24(X_men_train, y_outcome_men, y_team_men, y_opp_men,
                               X_men_test, gender='men')
print(f'✅ Men model done in {time.time()-t0:.1f}s')

In [ ]:
# ============================================================
# CELL 6: Train & Predict — WOMEN (V24)
# ============================================================
print('\n' + '='*60)
print('TRAINING WOMEN MODEL (V24)')
print('='*60)

X_women_train = train_women[FEATURE_COLS].values.astype(np.float32)
y_outcome_women = train_women['outcome'].values
y_team_women = np.clip(train_women['team_goals'].values.astype(int), 0, 5)
y_opp_women = np.clip(train_women['opp_goals'].values.astype(int), 0, 5)

X_women_test = test_women[FEATURE_COLS].values.astype(np.float32)

t0 = time.time()
women_results = train_model_v24(X_women_train, y_outcome_women, y_team_women, y_opp_women,
                                  X_women_test, gender='women')
print(f'✅ Women model done in {time.time()-t0:.1f}s')

In [ ]:
# ============================================================
# CELL 7: Evaluate & Submission — V24
# ============================================================
print('\n' + '='*60)
print('EVALUATION — V24')
print('='*60)

all_pred_team = np.concatenate([men_results['team_goals'], women_results['team_goals']])
all_pred_opp = np.concatenate([men_results['opp_goals'], women_results['opp_goals']])

gt_men = ground_truth[ground_truth['id'].isin(test_men['id'].values)]
gt_women = ground_truth[ground_truth['id'].isin(test_women['id'].values)]
gt_men = gt_men.set_index('id').loc[test_men['id'].values].reset_index()
gt_women = gt_women.set_index('id').loc[test_women['id'].values].reset_index()

true_team_all = np.concatenate([gt_men['team_goals'].values, gt_women['team_goals'].values])
true_opp_all = np.concatenate([gt_men['opp_goals'].values, gt_women['opp_goals'].values])

n_men = len(gt_men)

aw_mae = compute_aw_mae(all_pred_team, all_pred_opp, true_team_all, true_opp_all)
aw_mae_men = compute_aw_mae(all_pred_team[:n_men], all_pred_opp[:n_men], true_team_all[:n_men], true_opp_all[:n_men])
aw_mae_women = compute_aw_mae(all_pred_team[n_men:], all_pred_opp[n_men:], true_team_all[n_men:], true_opp_all[n_men:])
outcome_acc = compute_outcome_acc(all_pred_team, all_pred_opp, true_team_all, true_opp_all)
exact_acc = compute_exact_acc(all_pred_team, all_pred_opp, true_team_all, true_opp_all)

print(f'\n📊 V24 Results:')
print(f'  Overall AW-MAE:    {aw_mae:.5f}')
print(f'  Men AW-MAE:        {aw_mae_men:.5f}')
print(f'  Women AW-MAE:      {aw_mae_women:.5f}')
print(f'  Outcome Accuracy:  {outcome_acc:.4f} ({outcome_acc*100:.1f}%)')
print(f'  Exact Score Acc:   {exact_acc:.4f} ({exact_acc*100:.1f}%)')

submission = pd.DataFrame({
    'id': np.concatenate([test_men['id'].values, test_women['id'].values]),
    'pred_team_goals': all_pred_team.astype(int),
    'pred_opp_goals': all_pred_opp.astype(int)
})
submission.to_csv('submission_v24.csv', index=False)
print(f'\n✅ Submission saved to submission_v24.csv ({len(submission)} rows)')

## ✅ V24 Complete!

**V24 Strategy Summary:**
- ✅ Gender-Split 100%
- ✅ Isotonic Calibration Stage 1 (CV-based)
- ✅ LGBM + CatBoost Ensemble Stage 2
- ✅ Soft Cascade Renormalization
- ✅ ERM Decision Rule
- ✅ Temperature Scaling (T=1.1 Men, T=1.2 Women)

Hasil submission: **submission_v24.csv**